# Create zarr stroring unstitched tiles

Related image.sc thread: https://forum.image.sc/t/separate-tiles-to-ome-zarr/109071/60

Ome-zarr libs: https://github.com/ome/ome-zarr-py/issues/407

In [ ]:
from enum import Enum
import glob
import numpy as np
from ome_zarr.image import NgffImage, NgffMultiscales
from ome_zarr.scene import NgffScene
from ome_zarr_models._v06.coordinate_transforms import CoordinateSystem, CoordinateSystemIdentifier, Axis, Translation
import os.path
import tifffile

## Source data

In [ ]:
source_path = 'D:/slides/example_cells/*.tiff'

filenames = glob.glob(source_path)
filenames

## Functions to read tiff, convert to ome-zarr and write scene

In [ ]:
def read_tiff(filename):
    tif = tifffile.TiffFile(filename)
    data = tif.asarray()
    data = rgba_to_rgb(data)
    data = np.moveaxis(data, -1, 0)     # move channel dimension from last to first position
    metadata = tags_to_dict(tif.pages.first.tags)
    metadata['position'] = {
        'y': convert_rational_value(metadata.get('YPosition')),
        'x': convert_rational_value(metadata.get('XPosition'))
    }
    return data, metadata


def tags_to_dict(tags):
    tag_dict = {}
    for tag in tags.values():
        value = tag.value
        if isinstance(value, Enum):
            value = value.name
        tag_dict[tag.name] = value
    return tag_dict


def convert_rational_value(value):
    if value is not None and isinstance(value, (tuple, list)):
        if value[0] == value[1]:
            value = value[0]
        else:
            value = value[0] / value[1]
    return value


def get_stage_position(tile_name, shape, pixel_size):
    pixel_size_list = np.array(list(pixel_size.values()))
    position = (np.array(list(map(int, tile_name.split('_')[:2]))) * shape[1:] * pixel_size_list).tolist()
    return {'y': position[0], 'x': position[1]}


def get_aligned_position(metadata, pixel_size):
    position_pixels = metadata['position']
    position = {dim: pixels * pixel_size.get(dim, 1) for dim, pixels in position_pixels.items()}
    return position


def rgba_to_rgb(data):
    alpha = np.atleast_3d(data[:, :, 3] / np.float32(255))
    return (data[:, :, :3] * alpha).astype(np.uint8)


def init_cs(dim_order):
    axes = []
    for dim in dim_order:
        if dim == 'c':
            axis = Axis(name=dim, type='channel')
        elif dim == 't':
            axis = Axis(name=dim, type='time', unit='second')
        else:
            axis = Axis(name=dim, type='space', unit='micrometer')
        axes.append(axis)

    cs_world = CoordinateSystem(
        name="stitched",
        axes=tuple(axes)
    )
    return cs_world


def write_tile(path, data, dim_order, pixel_size, name):
    scale_factors = [pyramid_downsample ** (scale + 1) for scale in range(npyramid_add)]

    ngff_image = NgffImage(data, dim_order, pixel_size, {'x': 'micrometer', 'y': 'micrometer'}, name=name)
    ngff_multiscales = NgffMultiscales(ngff_image, scale_factors)

    ngff_multiscales.to_ome_zarr(path)

    return ngff_multiscales


def create_coordinate_transform(multiscales, translation, dim_order):
    if isinstance(translation, dict):
        translation = [translation.get(dim, 0) for dim in dim_order]
    while len(translation) < len(dim_order):
        translation = [0] + list(translation)
    transform = Translation(
        translation=translation,
        input=CoordinateSystemIdentifier(
                path=multiscales.name,
                name=multiscales.metadata.coordinateSystems[0].name
            ),
        output=CoordinateSystemIdentifier(name="stitched")
    )
    return transform


def write_scene(path, images, transforms, cs_world):
    ngff_scene = NgffScene(
        images=images,
        transformations=transforms,
        coordinate_systems=(cs_world,)
    )

    ngff_scene.to_ome_zarr(path)

## Read source images and store as ome-zarr

In [ ]:
pyramid_downsample = 2
npyramid_add = 2
zarr_version = 3
ome_version = '0.6'
dim_order = 'cyx'
pixel_size = {'y': 0.004, 'x': 0.004}

output_path = '../output/tiles.ome.zarr'


cs = init_cs(dim_order)

images = []
transforms = []
for filename in filenames:
    data, metadata = read_tiff(filename)
    tile_name = os.path.splitext(os.path.basename(filename))[0]
    path = f'{output_path}/{tile_name}'
    translation0 = get_stage_position(tile_name, data.shape, pixel_size)
    translation = get_aligned_position(metadata, pixel_size)
    multiscales = write_tile(path, data, dim_order, pixel_size, tile_name)
    images.append(multiscales)
    transforms.append(create_coordinate_transform(multiscales, translation, dim_order))
    print(f'{filename} -> {path} @ {list(translation.values())}')
print('writing scene')
write_scene(output_path, images, transforms, cs)
print('done')
